# SGB — Sistema de Gestión de Biblioteca
Proyecto POO · Python 3.10+ · Google Colab · Arquitectura de 3 capas (Modelo, Servicios, Repositorios)

## Celda 0 — Configuración

In [ ]:
from google.colab import drive
import os

# Montar Google Drive
drive.mount('/content/drive')

# Ruta base del proyecto en Drive
RUTA_BASE = '/content/drive/MyDrive/SGB/datos/'
os.makedirs(RUTA_BASE, exist_ok=True)

# Rutas de cada archivo de persistencia
RUTA_USUARIOS   = RUTA_BASE + 'usuarios.txt'
RUTA_LIBROS     = RUTA_BASE + 'libros.txt'
RUTA_CATEGORIAS = RUTA_BASE + 'categorias.txt'
RUTA_PRESTAMOS  = RUTA_BASE + 'prestamos.txt'
RUTA_MULTAS     = RUTA_BASE + 'multas.txt'
RUTA_RESERVAS   = RUTA_BASE + 'reservas.txt'

# Constantes de negocio
SEPARADOR = '|'
MULTA_POR_DIA = 1000  # COP por cada día de retraso

## Sección 1 — Modelo (dominio)

### Enums

In [ ]:
from enum import Enum


class TipoUsuario(Enum):
    """Tipo de usuario con su plazo de préstamo (días) y cupo de libros."""
    ESTUDIANTE = ("estudiante", 7, 3)
    DOCENTE = ("docente", 15, 5)
    EXTERNO = ("externo", 5, 2)

    def __init__(self, etiqueta, dias_prestamo, cupo):
        self.etiqueta = etiqueta
        self.dias_prestamo = dias_prestamo
        self.cupo = cupo

    @classmethod
    def desde_texto(cls, texto):
        """Convierte un texto ('estudiante', 'docente', 'externo') en TipoUsuario."""
        for t in cls:
            if t.etiqueta == texto.strip().lower():
                return t
        raise ValueError(f"Tipo de usuario inválido: {texto}")


class EstadoUsuario(Enum):
    """Estado de un usuario en el sistema."""
    ACTIVO = "activo"
    SUSPENDIDO = "suspendido"


class EstadoReserva(Enum):
    """Estado de una reserva de libro."""
    ACTIVA = "activa"
    CANCELADA = "cancelada"
    ATENDIDA = "atendida"

### Usuario y Categoria

In [ ]:
class Usuario:
    """Usuario de la biblioteca (estudiante, docente o externo)."""

    def __init__(self, documento, nombre, tipo, correo, telefono,
                 estado=EstadoUsuario.ACTIVO):
        self.documento = documento
        self.nombre = nombre
        self.tipo = tipo
        self.correo = correo
        self.telefono = telefono
        self.estado = estado

    def esta_activo(self):
        """Retorna True si el usuario está activo."""
        return self.estado == EstadoUsuario.ACTIVO

    def to_linea(self):
        """Serializa el usuario a una línea delimitada por pipe."""
        return SEPARADOR.join([self.documento, self.nombre, self.tipo.etiqueta,
                               self.correo, self.telefono, self.estado.value])

    @classmethod
    def desde_linea(cls, linea):
        """Construye un Usuario desde una línea del archivo."""
        d, n, t, c, tel, e = linea.strip().split(SEPARADOR)
        return cls(d, n, TipoUsuario.desde_texto(t), c, tel, EstadoUsuario(e))

    def __str__(self):
        return (f"{self.documento} | {self.nombre} | {self.tipo.etiqueta} | "
                f"{self.correo} | {self.telefono} | {self.estado.value}")


class Categoria:
    """Categoría temática de libros."""

    def __init__(self, id_categoria, nombre):
        self.id_categoria = id_categoria
        self.nombre = nombre

    def to_linea(self):
        """Serializa la categoría a una línea delimitada por pipe."""
        return SEPARADOR.join([str(self.id_categoria), self.nombre])

    @classmethod
    def desde_linea(cls, linea):
        """Construye una Categoria desde una línea del archivo."""
        i, n = linea.strip().split(SEPARADOR)
        return cls(int(i), n)

    def __str__(self):
        return f"[{self.id_categoria}] {self.nombre}"

### Libro

In [ ]:
class Libro:
    """Libro del catálogo con control de ejemplares totales y disponibles."""

    def __init__(self, isbn, titulo, autor, id_categoria, anio,
                 ejemplares_totales, ejemplares_disponibles=None):
        self.isbn = isbn
        self.titulo = titulo
        self.autor = autor
        self.id_categoria = id_categoria
        self.anio = anio
        self.ejemplares_totales = ejemplares_totales
        self.ejemplares_disponibles = (ejemplares_totales
                                       if ejemplares_disponibles is None
                                       else ejemplares_disponibles)

    def hay_disponibles(self):
        """Retorna True si hay al menos un ejemplar disponible."""
        return self.ejemplares_disponibles > 0

    def prestar_ejemplar(self):
        """Descuenta un ejemplar disponible."""
        if not self.hay_disponibles():
            raise ValueError("No hay ejemplares disponibles")
        self.ejemplares_disponibles -= 1

    def devolver_ejemplar(self):
        """Suma un ejemplar disponible (sin superar el total)."""
        if self.ejemplares_disponibles < self.ejemplares_totales:
            self.ejemplares_disponibles += 1

    def to_linea(self):
        """Serializa el libro a una línea delimitada por pipe."""
        return SEPARADOR.join([self.isbn, self.titulo, self.autor,
                               str(self.id_categoria), str(self.anio),
                               str(self.ejemplares_totales),
                               str(self.ejemplares_disponibles)])

    @classmethod
    def desde_linea(cls, linea):
        """Construye un Libro desde una línea del archivo."""
        i, t, a, c, an, tot, disp = linea.strip().split(SEPARADOR)
        return cls(i, t, a, int(c), int(an), int(tot), int(disp))

    def __str__(self):
        return (f"{self.isbn} | {self.titulo} | {self.autor} | cat {self.id_categoria} | "
                f"{self.anio} | {self.ejemplares_disponibles}/{self.ejemplares_totales}")

### Prestamo, Multa y Reserva

In [ ]:
from datetime import date, timedelta


class Prestamo:
    """Préstamo de un libro a un usuario."""

    def __init__(self, id_prestamo, documento, isbn, fecha_prestamo,
                 fecha_limite, fecha_devolucion=None):
        self.id_prestamo = id_prestamo
        self.documento = documento
        self.isbn = isbn
        self.fecha_prestamo = fecha_prestamo
        self.fecha_limite = fecha_limite
        self.fecha_devolucion = fecha_devolucion

    @staticmethod
    def calcular_fecha_limite(fecha_prestamo, tipo_usuario):
        """Fecha límite según el tipo de usuario (7, 15 o 5 días)."""
        return fecha_prestamo + timedelta(days=tipo_usuario.dias_prestamo)

    def esta_activo(self):
        """True si el libro aún no ha sido devuelto."""
        return self.fecha_devolucion is None

    def esta_vencido(self, fecha_referencia):
        """True si está activo y la fecha de referencia supera la fecha límite."""
        return self.esta_activo() and fecha_referencia > self.fecha_limite

    def dias_retraso(self):
        """Días de retraso al devolver (0 si no hubo o no se ha devuelto)."""
        if self.fecha_devolucion is None:
            return 0
        return max(0, (self.fecha_devolucion - self.fecha_limite).days)

    def duracion_real(self):
        """Duración real del préstamo en días (None si sigue activo)."""
        if self.fecha_devolucion is None:
            return None
        return (self.fecha_devolucion - self.fecha_prestamo).days

    def to_linea(self):
        """Serializa el préstamo a una línea delimitada por pipe."""
        dev = self.fecha_devolucion.isoformat() if self.fecha_devolucion else ""
        return SEPARADOR.join([str(self.id_prestamo), self.documento, self.isbn,
                               self.fecha_prestamo.isoformat(),
                               self.fecha_limite.isoformat(), dev])

    @classmethod
    def desde_linea(cls, linea):
        """Construye un Prestamo desde una línea del archivo."""
        i, d, isbn, fp, fl, fd = linea.rstrip("\n").split(SEPARADOR)
        return cls(int(i), d, isbn, date.fromisoformat(fp),
                   date.fromisoformat(fl),
                   date.fromisoformat(fd) if fd else None)

    def __str__(self):
        dev = self.fecha_devolucion or "pendiente"
        return (f"#{self.id_prestamo} | usuario {self.documento} | libro {self.isbn} | "
                f"{self.fecha_prestamo} -> límite {self.fecha_limite} | devuelto: {dev}")


class Multa:
    """Multa generada por devolución tardía."""

    def __init__(self, id_multa, id_prestamo, documento, dias_retraso,
                 valor, pagada=False):
        self.id_multa = id_multa
        self.id_prestamo = id_prestamo
        self.documento = documento
        self.dias_retraso = dias_retraso
        self.valor = valor
        self.pagada = pagada

    @staticmethod
    def calcular_valor(dias_retraso):
        """Valor de la multa: COP 1.000 por día de retraso."""
        return dias_retraso * MULTA_POR_DIA

    def pagar(self):
        """Marca la multa como pagada."""
        self.pagada = True

    def to_linea(self):
        """Serializa la multa a una línea delimitada por pipe."""
        return SEPARADOR.join([str(self.id_multa), str(self.id_prestamo),
                               self.documento, str(self.dias_retraso),
                               str(self.valor), "1" if self.pagada else "0"])

    @classmethod
    def desde_linea(cls, linea):
        """Construye una Multa desde una línea del archivo."""
        i, p, d, dr, v, pg = linea.strip().split(SEPARADOR)
        return cls(int(i), int(p), d, int(dr), int(v), pg == "1")

    def __str__(self):
        estado = "pagada" if self.pagada else "pendiente"
        return (f"Multa #{self.id_multa} | préstamo {self.id_prestamo} | "
                f"{self.dias_retraso} días | ${self.valor:,} COP | {estado}")


class Reserva:
    """Reserva de un libro sin ejemplares disponibles (cola por orden de llegada)."""

    def __init__(self, id_reserva, documento, isbn, fecha,
                 estado=EstadoReserva.ACTIVA):
        self.id_reserva = id_reserva
        self.documento = documento
        self.isbn = isbn
        self.fecha = fecha
        self.estado = estado

    def cancelar(self):
        """Cancela la reserva."""
        self.estado = EstadoReserva.CANCELADA

    def atender(self):
        """Marca la reserva como atendida."""
        self.estado = EstadoReserva.ATENDIDA

    def to_linea(self):
        """Serializa la reserva a una línea delimitada por pipe."""
        return SEPARADOR.join([str(self.id_reserva), self.documento, self.isbn,
                               self.fecha.isoformat(), self.estado.value])

    @classmethod
    def desde_linea(cls, linea):
        """Construye una Reserva desde una línea del archivo."""
        i, d, isbn, f, e = linea.strip().split(SEPARADOR)
        return cls(int(i), d, isbn, date.fromisoformat(f), EstadoReserva(e))

    def __str__(self):
        return (f"Reserva #{self.id_reserva} | usuario {self.documento} | "
                f"libro {self.isbn} | {self.fecha} | {self.estado.value}")

## Prueba rápida del modelo

In [ ]:
# Prueba rápida (se puede borrar en el siguiente commit)
u = Usuario("1098765432", "Ana Pérez", TipoUsuario.ESTUDIANTE, "ana@mail.com", "3001234567")
l = Libro("978-0134685991", "Effective Java", "Joshua Bloch", 1, 2018, 3)
hoy = date.today()
p = Prestamo(1, u.documento, l.isbn, hoy, Prestamo.calcular_fecha_limite(hoy, u.tipo))
print(u); print(l); print(p)
assert Usuario.desde_linea(u.to_linea()).nombre == u.nombre
assert Prestamo.desde_linea(p.to_linea()).fecha_limite == p.fecha_limite
assert Multa.calcular_valor(3) == 3000
print("✔ Modelo OK")